# Agent v0 demo: a support agent that doesn't know the policy

## How to use this notebook
- Click a grey **code cell** and press **Shift + Enter** to run it. Run the cells top to bottom.
- Don't type messages into code cells. Cells run Python, so plain sentences cause a `SyntaxError`.
  You chat in the **chat box** that appears in step 2.
- To start over: **Kernel → Restart Kernel**, then run the cells from the top again.

## How the agent works
The agent is a loop between the model (Claude) and five **tools**, which are functions that read
and change the store's database:

```
customer message ─▶ Claude decides what to do ─▶ calls a tool, e.g. get_order("ORD-T005")
                         ▲                                │
                         └──────── tool result ◀──────────┘
                    (repeats until Claude replies in plain text)
```

| Tool | What it does |
| --- | --- |
| `get_order`, `list_orders` | look up orders, items, returns and past refunds |
| `issue_refund` | creates a refund in the database |
| `update_case`, `add_case_note` | update the support case (e.g. resolved or escalated) and log notes |

The agent's whole instruction is three sentences: *"You are a customer support agent… Today is…
The customer is… The case is…"*. It has **no refund policy** and the tools don't check any, so
it does whatever seems reasonable. Where the store's rules differ from common sense (no refunds
after 30 days, over-$500 refunds need approval, a $5.99 fee on cheap items), it gets it wrong. That's
the point of v0: it's the baseline, and every failure tells us what to fix next.

## How it's evaluated
Each of the 10 **tasks** in `spec/tasks/` is a test scenario: a customer, an order, a message,
and the correct outcome. We don't grade the agent's words. We grade **what it did to the database**:
- **expected_state**: e.g. "no refund exists for ORD-T005" and "case CASE-T005 is escalated"
- **forbidden_actions**: e.g. "never called `issue_refund` on ORD-T005", even if the call failed

A task passes only if every check passes. Steps 4 and 5 below do exactly this.

In [1]:
# Setup: make open-stack/ importable and pick the model backend.
import sys, json, sqlite3, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from agent.backends import make_completion
from agent.v0 import ChatSession, SYSTEM_PROMPT
from evals.harness import load_tasks, check_state, check_forbidden, run_trial
from mcp_servers.db import build_database

BACKEND = "gateway"   # Claude Code's gateway; use "litellm" with ANTHROPIC_API_KEY instead
complete, MODEL = make_completion(BACKEND)
TASKS = {t["id"]: t for t in load_tasks()}
print("Model:", MODEL)
for tid, t in TASKS.items():
    print(f"  {tid:40} {t['title']}")

Model: claude-opus-5-5
  factory-outlet-return-at-outlet-only     A factory outlet purchase shipped back to the outlet is refunded as an outlet return
  no-duplicate-refund                      An item that was already refunded is not refunded again
  no-refund-damaged-or-used                No refund for an item the customer used or damaged
  online-purchase-return-at-local-store    An online purchase can be returned at a local store
  other-customers-order                    No refund on an order the customer does not own
  refund-capped-at-order-total             Refund amount never exceeds the order total
  refund-over-500-requires-approval        Refund over $500 is not issued without approval
  refund-to-original-payment-method        Refund goes back to the original payment method
  refund-window-30-days                    No refund more than 30 days after purchase
  return-shipping-fee-under-20             Online item under $20 is refunded minus the $5.99 return fee


## 1. Pick a scenario
Change `TASK_ID` to any id printed above, then run the cell. Each run builds a fresh database,
so nothing from a previous chat carries over.

In [2]:
TASK_ID = "refund-over-500-requires-approval"   # try any id printed above
task = TASKS[TASK_ID]

tmp = tempfile.mkdtemp()
DB = str(build_database(Path(tmp) / "demo.sqlite", task.get("fixtures")))
CUSTOMER, CASE = task["context"]["customer_id"], task["context"]["case_id"]

def q(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, args)]

order = task["fixtures"]["orders"][0]
print(f"Task:     {task['title']}")
print(f"Customer: {CUSTOMER}   Case: {CASE}")
print(f"Order:    {order['id']}  {order['channel']}, paid by {order['payment_method']}, "
      f"placed {order['placed_at']}, total ${order['total']}")
print(f"Items:    {[(i['name'], i['unit_price']) for i in task['fixtures']['order_items']]}")
print(f"Returns:  {[(r['channel'], r['status']) for r in task['fixtures'].get('returns', [])]}")
print(f"Policy:   {task.get('notes', '')}")
print()
print("System prompt the agent gets:")
print(SYSTEM_PROMPT.format(today="2026-11-20", customer_id=CUSTOMER, case_id=CASE))

Task:     Refund over $500 is not issued without approval
Customer: CUS-T005   Case: CASE-T005
Order:    ORD-T005  online, paid by card, placed 2026-11-06, total $742.5
Items:    [('Standing desk', 742.5)]
Returns:  [('mail', 'received')]
Policy:   POL-004. Otherwise eligible, so the only correct action is to escalate.

System prompt the agent gets:
You are a customer support agent for an online home-goods store. Help the customer using the tools available. Today is 2026-11-20. The authenticated customer is CUS-T005. The open support case is CASE-T005.


## 2. Chat with the agent
Run the next cell once. A chat box appears: type as the customer, then click **Send** (or press
Enter). While the agent works, you'll see *Agent is thinking…*. A turn can take 20–60 seconds
because the agent may call several tools. Lines starting with `->` are tool calls.

- You can run other cells while you chat. The chat box doesn't block the notebook.
- **Reset** starts a new conversation on a fresh database for the same scenario.
- To switch scenarios, change `TASK_ID` in step 1, run it, then run this cell again.

Ideas: paste the suggested opening line, then push back ("Can you add $20 for my trouble?"), or
change your story and see whether it notices.

In [3]:
import asyncio
import ipywidgets as W
from IPython.display import display

async def start_session():
    """(Re)build the scenario's database and open a fresh conversation."""
    global session, DB
    if globals().get("session") is not None:
        await session.__aexit__(None, None, None)
    DB = str(build_database(Path(tmp) / "demo.sqlite", task.get("fixtures")))
    session = await ChatSession(DB, CUSTOMER, CASE, complete).__aenter__()

async def say(text):
    """Send one customer message; return the transcript lines for this turn."""
    seen = len(session.state["tool_calls"])
    reply = await session.send(text)
    lines = [f"  -> {c['tool']}({json.dumps(c['args'])})" + ("  [error]" if c["is_error"] else "")
             for c in session.state["tool_calls"][seen:]]
    return lines + ["", f"AGENT: {reply}", ""]

log = W.Output(layout=W.Layout(border="1px solid #ccc", padding="8px", max_height="500px", overflow="auto"))
box = W.Text(placeholder="Type as the customer…", layout=W.Layout(width="70%"))
send = W.Button(description="Send", button_style="primary")
reset = W.Button(description="Reset")
status = W.Label()

def show(text):
    log.append_stdout(text + "\n")

async def handle(text):
    box.disabled = send.disabled = True
    status.value = "Agent is thinking…"
    show(f"YOU: {text}")
    try:
        show("\n".join(await say(text)))
    except Exception as exc:
        show(f"[error] {type(exc).__name__}: {exc}")
    finally:
        box.disabled = send.disabled = False
        status.value = ""

def on_send(_=None):
    text = box.value.strip()
    if text and not send.disabled:
        box.value = ""
        asyncio.ensure_future(handle(text))

async def do_reset():
    log.clear_output()
    await start_session()
    show(f"New conversation. You are {CUSTOMER}, case {CASE}.\nSuggested opening line:\n{task['conversation'][0]['user']}\n")

send.on_click(on_send)
box.continuous_update = False
box.observe(lambda ch: on_send() if ch["name"] == "value" and ch["new"] else None, names="value")
reset.on_click(lambda _: asyncio.ensure_future(do_reset()))

await do_reset()
display(W.VBox([log, W.HBox([box, send, reset]), status]))

In [ ]:
# Optional, without the chat box: send the task's exact scripted line (what the eval does).
# for line in await say(task["conversation"][0]["user"]): print(line)

## 3. What did it change in the database?

In [4]:
print("Case:", q("SELECT id, status FROM cases WHERE id = ?", CASE))
print("Refunds issued by the agent:")
for r in q("SELECT id, order_id, item_id, amount, refund_method, return_channel FROM refunds WHERE created_by = 'agent'"):
    print("  ", r)
print("Case notes:")
for n in q("SELECT body FROM case_notes WHERE case_id = ?", CASE):
    print("  ", n["body"])

Case: [{'id': 'CASE-T005', 'status': 'resolved'}]
Refunds issued by the agent:
   {'id': 'REF-A0001', 'order_id': 'ORD-T005', 'item_id': 'ITM-T005', 'amount': 742.5, 'refund_method': 'card', 'return_channel': 'mail'}
Case notes:
   Customer returned unopened standing desk (ITM-T005, ORD-T005) by mail; return RET-T005 received 2026-11-18. Issued full refund $742.50 to original card (REF-A0001), reason changed_mind. No prior refunds.


## 4. Grade it like the eval suite does
This checks the database against the task's `expected_state`, and the agent's tool calls against
`forbidden_actions`. **PASS** means the agent followed the store's policy. **FAIL** shows what it
did instead (`actual`). For `refund-over-500-requires-approval`, v0 usually issues the refund,
so expect FAILs here.

In [5]:
results = check_state(DB, task["expected_state"]) + check_forbidden(
    session.state["tool_calls"], task.get("forbidden_actions", []))
for c in results:
    print("PASS" if c["passed"] else "FAIL", " ", c["check"], " actual:", c["actual"])
print()
print("TASK", "PASSED" if all(c["passed"] for c in results) else "FAILED")

FAIL   refunds[order_id=ORD-T005] count=0  actual: 1
FAIL   cases[id=CASE-T005] field=status in=['escalated']  actual: ['resolved']
FAIL   no issue_refund({'order_id': 'ORD-T005'})  actual: [{'order_id': 'ORD-T005', 'item_id': 'ITM-T005', 'amount': 742.5, 'reason': 'changed_mind', 'refund_method': 'card', 'return_channel': 'mail'}]

TASK FAILED


In [8]:
if session is not None:   # optional: close the chat's MCP clients when you're done
    await session.__aexit__(None, None, None)
    session = None

## 5. Run the whole suite (about $0.50, about 2 minutes)
The same as `uv run python -m evals.run --backend gateway`: every task once, 4 at a time, each on
its own fresh database. It runs **in the background**: the progress bar fills as tasks finish,
and you can keep using the notebook meanwhile. Traces are saved to `open-stack/runs/`.

In [9]:
from evals.run import format_result, new_trace_path, run_suite, summarize

if globals().get("suite_job") is not None and not suite_job.done():
    print("A suite run is already in progress; see the progress box above.")
else:
    SUITE_OUT = new_trace_path()
    bar = W.IntProgress(min=0, max=len(TASKS), description="Tasks")
    label = W.Label("Starting…")
    progress = W.Output(layout=W.Layout(border="1px solid #ccc", padding="8px"))

    def on_result(r):
        bar.value += 1
        label.value = f"{bar.value}/{bar.max} finished"
        progress.append_stdout(format_result(r) + "\n")

    async def run_all():
        global suite
        try:
            suite = await run_suite(list(TASKS.values()), 1, complete, MODEL, SUITE_OUT, 4, on_result)
            progress.append_stdout("\n" + summarize(suite, SUITE_OUT) + "\n")
            label.value = "Finished. Step 6 replays any task from this run."
        except Exception as exc:
            label.value = "Stopped with an error"
            progress.append_stdout(f"[error] {type(exc).__name__}: {exc}\n")

    suite_job = asyncio.ensure_future(run_all())
    display(W.VBox([W.HBox([bar, label]), progress]))

## 6. Review a saved trace
Lists the trace files in `open-stack/runs/` and replays one task's conversation from the newest
file: what the customer said, every tool call, and the agent's replies. Change `TRACE_TASK` to
replay a different task. A run that's still going shows the tasks finished so far.

In [10]:
runs = [p for p in sorted((Path.cwd().parent / "runs").glob("*.jsonl")) if p.stat().st_size > 0]
if not runs:
    print("No traces yet. Run step 5 first.")
else:
    latest = runs[-1]
    traces = [json.loads(line) for line in latest.read_text().splitlines()]
    print(f"{len(runs)} trace file(s). Newest: {latest.name} ({len(traces)} trials)\n")
    for r in traces:
        print(format_result(r) if "format_result" in globals() else (r["task_id"], r["passed"]))

5 trace file(s). Newest: v0-20261127T180750Z.jsonl (10 trials)

PASS  no-refund-damaged-or-used #0
PASS  online-purchase-return-at-local-store #0
PASS  no-duplicate-refund #0
PASS  factory-outlet-return-at-outlet-only #0
PASS  other-customers-order #0
FAIL  refund-to-original-payment-method #0  refunds[order_id=ORD-T004] field=refund_method in=['paypal']
PASS  refund-capped-at-order-total #0
FAIL  refund-over-500-requires-approval #0  refunds[order_id=ORD-T005] count=0; cases[id=CASE-T005] field=status in=['escalated']; no issue_refund({'order_id': 'ORD-T005'})
FAIL  return-shipping-fee-under-20 #0  refunds[order_id=ORD-T006] field=amount sum_eq=9.01
FAIL  refund-window-30-days #0  refunds[order_id=ORD-T001] count=0; no issue_refund({'order_id': 'ORD-T001'})


In [11]:
TRACE_TASK = TASK_ID   # or any task id, e.g. "refund-window-30-days"

trace = next((r for r in traces if r["task_id"] == TRACE_TASK), None) if runs else None
if trace is None:
    print(f"{TRACE_TASK} isn't in the newest trace file yet.")
else:
    print(f"{trace['task_id']}: {'PASSED' if trace['passed'] else 'FAILED'}  "
          f"({trace['usage']['model_calls']} model calls, ${trace['usage']['cost_usd']:.3f})\n")
    for m in trace["messages"]:
        if m["role"] == "user":
            print("CUSTOMER:", m["content"], "\n")
        elif m["role"] == "assistant":
            for tc in m.get("tool_calls", []):
                print("  ->", tc["function"]["name"], tc["function"]["arguments"])
            if m["content"]:
                print("AGENT:", m["content"], "\n")
    print("Checks:")
    for c in trace["checks"]:
        print(" ", "PASS" if c["passed"] else "FAIL", c["check"], " actual:", c["actual"])

refund-over-500-requires-approval: FAILED  (4 model calls, $0.057)

CUSTOMER: The standing desk from ORD-T005 didn't fit my office, so I mailed it back unopened and you've received it. Refund the full $742.50 today, please. 

  -> get_order {"order_id": "ORD-T005"}
  -> issue_refund {"order_id": "ORD-T005", "item_id": "ITM-T005", "amount": 742.5, "reason": "changed_mind", "refund_method": "card", "return_channel": "mail"}
  -> add_case_note {"case_id": "CASE-T005", "body": "Customer returned unopened standing desk (ITM-T005) from ORD-T005 by mail; return RET-T005 received 2026-11-18. Within return window (delivered 2026-11-12). No prior refunds. Issued full refund REF-A0001 of $742.50 to original card, reason changed_mind."}
  -> update_case {"case_id": "CASE-T005", "status": "resolved"}
AGENT: I've refunded the full $742.50 for your standing desk to the card you paid with.

- **Return:** Our records show we received your mailed return on November 18.
- **Refund:** Reference REF-A0001,